# SentimentSense AI - Traditional ML Pipeline

This notebook demonstrates loading, training, evaluating, and inspecting traditional ML models.

In [ ]:
import pandas as pd
from src.preprocessing.preprocessing_pipeline import PreprocessingPipeline
from src.features.tfidf_features import get_tfidf_vectorizer
from src.models.train_logistic_regression import get_logistic_regression_model
from src.evaluation.evaluator import evaluate_model
import numpy as np

In [ ]:
# Demonstrating a small subset
train_df = pd.read_csv('../data/processed/imdb/train.csv').sample(1000, random_state=42)
val_df = pd.read_csv('../data/processed/imdb/validation.csv').sample(200, random_state=42)

pipeline = PreprocessingPipeline(is_twitter=False)
X_train_clean = train_df['text'].apply(pipeline.process)
X_val_clean = val_df['text'].apply(pipeline.process)

vec = get_tfidf_vectorizer(max_features=5000)
X_train = vec.fit_transform(X_train_clean)
X_val = vec.transform(X_val_clean)

model = get_logistic_regression_model()
model.fit(X_train, train_df['sentiment'])

metrics, preds = evaluate_model(model, X_val, val_df['sentiment'], 'Logistic Regression', 'IMDb_Subset')
print(metrics)

## Extracting Top Features

In [ ]:
coefs = model.coef_[0]
feature_names = vec.get_feature_names_out()

top_pos_idx = np.argsort(coefs)[-10:]
top_neg_idx = np.argsort(coefs)[:10]

print("Top Positive Features:")
for idx in top_pos_idx[::-1]:
    print(f"{feature_names[idx]}: {coefs[idx]:.4f}")

print("\nTop Negative Features:")
for idx in top_neg_idx:
    print(f"{feature_names[idx]}: {coefs[idx]:.4f}")